###  Checkout Experiment & Growth Analytics System

Business Objective:
Evaluate whether Variant B of checkout should be rolled out to all users.

Leadership wants to know:
1. Should Variant B be rolled out?
2. Where are the biggest funnel leaks?
3. Which segments drive results?
4. What is next month's business impact?

North Star Metric:
Conversion Rate (Purchase / Eligible Sessions)

Supporting Metrics:
- Revenue per eligible session
- Funnel step conversion rates
- Segment-wise uplift
- 30-day incremental orders

### PART B – ETL & CURATED TABLES

In [1]:
import pandas as pd
import numpy as np
from scipy import stats

sessions = pd.read_csv("data/sessions.csv")
events = pd.read_csv("data/events.csv")
orders = pd.read_csv("data/orders.csv")
users = pd.read_csv("data/users.csv")

In [2]:
sessions = sessions.drop_duplicates("session_id")
sessions["channel"] = sessions["channel"].str.lower()
sessions["device"] = sessions["device"].fillna("unknown")

events["event_ts"] = pd.to_datetime(events["event_ts"], errors="coerce")
orders["net_amount"] = pd.to_numeric(orders["net_amount"], errors="coerce")

In [3]:
# 3. BUILD fact_sessions
# Funnel flags
funnel = events.pivot_table(
    index="session_id",
    columns="event_type",
    aggfunc="size",
    fill_value=0
)

# Convert counts to boolean
funnel = (funnel > 0).reset_index()

# Rename columns
rename_map = {
    "product_view": "has_productview",
    "add_to_cart": "has_addtocart",
    "begin_checkout": "has_begincheckout",
    "payment_attempt": "has_paymentattempt",
    "purchase": "has_purchase"
}

funnel = funnel.rename(columns=rename_map)

# Ensure all expected funnel columns exist
for col in rename_map.values():
    if col not in funnel.columns:
        funnel[col] = False

# Revenue per session
session_revenue = (
    orders.groupby("session_id")["net_amount"]
    .sum()
    .reset_index()
)

# Merge into fact table
fact_sessions = (
    sessions
    .merge(funnel, on="session_id", how="left")
    .merge(session_revenue, on="session_id", how="left")
)

# Fill missing funnel flags
funnel_cols = list(rename_map.values())
fact_sessions[funnel_cols] = fact_sessions[funnel_cols].fillna(False)
fact_sessions["net_amount"] = fact_sessions["net_amount"].fillna(0)

# Ensure boolean type
fact_sessions[funnel_cols] = fact_sessions[funnel_cols].astype(bool)

print("fact_sessions shape:", fact_sessions.shape)
fact_sessions.head()

fact_sessions shape: (9000, 15)


,session_id,user_id,session_start_ts,device,channel,campaign_id,variant,is_new_user,has_addtocart,has_begincheckout,landing_view,has_paymentattempt,has_productview,has_purchase,net_amount
0,S0000001,U001627,2025-10-26T15:56:31,web,organic,C025,A,1,False,False,True,False,False,False,0.0
1,S0000002,U001038,2025-12-09T21:56:53,mobile,paid_social,C037,NaN,0,False,False,True,False,False,False,0.0
2,S0000003,NaN,2025-11-24T21:24:21,web,organic,C034,NaN,0,False,False,True,False,True,False,0.0
3,S0000004,U001072,2025-12-02T06:51:38,mobile,organic,C013,NaN,0,False,False,True,False,False,False,0.0
4,S0000005,U001301,2025-11-28T16:03:05,mobile,paid_social,C004,NaN,1,False,False,True,False,True,False,0.0


In [4]:
# eligible function

eligible = fact_sessions[
    (fact_sessions["device"] == "web") &
    (fact_sessions["user_id"].notna()) &
    (fact_sessions["variant"].isin(["A","B"]))
].copy()

print("Eligible sessions:", eligible.shape[0])
print(eligible["variant"].value_counts())

Eligible sessions: 2972
variant
A    1509
B    1463
Name: count, dtype: int64


### PART C — FUNNEL ANALYSIS

In [5]:
# Funnel conversion

funnel_steps = [
    "has_productview",
    "has_addtocart",
    "has_begincheckout",
    "has_paymentattempt",
    "has_purchase"
]

funnel_summary = {}

for step in funnel_steps:
    rate = eligible[step].mean()
    print(step, round(rate*100,2), "%")

has_productview 62.21 %
has_addtocart 19.11 %
has_begincheckout 12.99 %
has_paymentattempt 12.99 %
has_purchase 9.22 %


In [6]:
# Step-to-step conversion
step_conv = {}

step_conv["Product → Cart"] = (
    eligible["has_addtocart"].sum() /
    eligible["has_productview"].sum()
)

step_conv["Cart → Checkout"] = (
    eligible["has_begincheckout"].sum() /
    eligible["has_addtocart"].sum()
)

step_conv["Checkout → Payment"] = (
    eligible["has_paymentattempt"].sum() /
    eligible["has_begincheckout"].sum()
)

step_conv["Payment → Purchase"] = (
    eligible["has_purchase"].sum() /
    eligible["has_paymentattempt"].sum()
)

{k: round(v*100,2) for k,v in step_conv.items()}

{'Product → Cart': np.float64(30.72),
 'Cart → Checkout': np.float64(67.96),
 'Checkout → Payment': np.float64(100.0),
 'Payment → Purchase': np.float64(70.98)}

In [7]:
# Pre-cart dropoff
pre_cart_dropoff = 1 - eligible["has_addtocart"].mean()
print("Pre-cart dropoff:", round(pre_cart_dropoff*100,2), "%")

Pre-cart dropoff: 80.89 %


#### **Funnel Insights:**
The largest drop-off occurs before Add-to-Cart,
with 80.89% of eligible sessions not progressing to cart.

Although 62.21% of users view a product, only 19.11% add it to cart.
This translates to a Product → Cart efficiency of 30.72%, meaning nearly 7 out of 10 product viewers exit before forming purchase intent.

This indicates that the primary friction lies at the product engagement stage — potentially related to pricing perception, value clarity, trust signals, or traffic quality — rather than checkout mechanics alone.

In [8]:
# Segment analysis
eligible_seg = eligible.merge(
    users[["user_id","segment"]],
    on="user_id",
    how="left"
)

segment_conv = (
    eligible_seg.groupby("segment")["has_purchase"]
    .mean()
    .sort_values(ascending=False)
)

print(segment_conv)

segment
premium    0.101485
value      0.091146
regular    0.090395
Name: has_purchase, dtype: float64


### PART D – A/B EXPERIMENT

In [9]:
ab = eligible.groupby("variant")["has_purchase"].agg(["mean","count","sum"])
print(ab)

lift = (ab.loc["B","mean"] - ab.loc["A","mean"]) / ab.loc["A","mean"]
print("Lift %:", round(lift*100,2))

             mean  count  sum
variant                      
A        0.084162   1509  127
B        0.100478   1463  147
Lift %: 19.39


In [10]:
# chi square
from scipy import stats

contingency = pd.crosstab(eligible["variant"], eligible["has_purchase"])
chi2, p, _, _ = stats.chi2_contingency(contingency)

print("p-value:", p)

p-value: 0.1405401562399662


#### **Experiment Conclusion:**
Variant B improved conversion from 8.42% to 10.05%,
representing a +19.39% relative lift.

However, the result is not statistically significant
(Chi-square p = 0.1405 > 0.05).

While Variant B shows promising directional improvement, the current sample size does not provide sufficient statistical confidence to support a full rollout decision.

Recommendation:
Continue the experiment to increase statistical power before a global rollout. In parallel, consider a targeted rollout within high-performing segments (e.g., Premium users), where baseline conversion is strongest and potential upside may be higher.

In [11]:
# HTE: Variant performance by segment

eligible_seg = eligible.merge(
    users[["user_id","segment"]],
    on="user_id",
    how="left"
)

hte_segment = (
    eligible_seg
    .groupby(["segment","variant"])["has_purchase"]
    .mean()
    .unstack()
)

hte_segment["lift_%"] = (
    (hte_segment["B"] - hte_segment["A"]) /
    hte_segment["A"]
) * 100

hte_segment

variant,A,B,lift_%
segment,,,
premium,0.088235,0.115000,30.333333
regular,0.084270,0.096591,14.621212
value,0.082631,0.100179,21.236903


In [12]:
# HTE: New vs Returning

hte_new = (
    eligible
    .groupby(["is_new_user","variant"])["has_purchase"]
    .mean()
    .unstack()
)

hte_new["lift_%"] = (
    (hte_new["B"] - hte_new["A"]) /
    hte_new["A"]
) * 100

hte_new

variant,A,B,lift_%
is_new_user,,,
0,0.089407,0.100287,12.168307
1,0.072917,0.100962,38.461538


In [13]:
# HTE: Channel-level performance

hte_channel = (
    eligible
    .groupby(["channel","variant"])["has_purchase"]
    .mean()
    .unstack()
)

hte_channel["lift_%"] = (
    (hte_channel["B"] - hte_channel["A"]) /
    hte_channel["A"]
) * 100

hte_channel

variant,A,B,lift_%
channel,,,
email,0.087805,0.107955,22.948232
organic,0.085389,0.089431,4.733514
paid_social,0.089494,0.096654,8.000647
referral,0.089744,0.112583,25.449385
search,0.074176,0.109333,47.397531


#### **Heterogeneous Treatment Effects (HTE) Insight:**

Variant B demonstrates heterogeneous performance across user groups.

The strongest uplift is observed among:
- New users (+38%)
- Search channel users (+47%)
- Premium segment users (+30%)
- More modest uplift is observed among:
- Regular segment (+14%)
- Paid Social (+8%)
- Organic traffic (+4.7%)

These findings suggest that Variant B disproportionately benefits high-intent and first-time users, indicating checkout friction reduction is most impactful where purchase hesitation is highest.

While overall statistical significance was not achieved (p = 0.1405), the consistent directional improvement across segments — particularly among high-value cohorts — supports consideration of a targeted rollout strategy rather than a full global deployment.

### PART E – IMPACT ESTIMATION

In [14]:
# PART E – Business Impact Projection

# Current metrics
conv_A = ab.loc["A","mean"]
conv_B = ab.loc["B","mean"]

total_eligible = eligible.shape[0]

# Current blended conversion
current_conv = eligible["has_purchase"].mean()

# Expected if full rollout of B
expected_conv = conv_B

# Incremental conversion
incremental_conv = expected_conv - current_conv

# Incremental orders
incremental_orders = incremental_conv * total_eligible

print("Current conversion:", round(current_conv*100,2), "%")
print("Expected conversion (if B rolled out):", round(expected_conv*100,2), "%")
print("Incremental orders (per similar traffic period):", round(incremental_orders))

Current conversion: 9.22 %
Expected conversion (if B rolled out): 10.05 %
Incremental orders (per similar traffic period): 25


In [15]:
# Average order value (revenue impact)
avg_order_value = orders["net_amount"].mean()

incremental_revenue = incremental_orders * avg_order_value

print("Average Order Value:", round(avg_order_value,2))
print("Projected incremental revenue:", round(incremental_revenue,2))

Average Order Value: 7738.8
Projected incremental revenue: 190544.85


In [16]:
# Check dataset span using eligible sessions

eligible["session_start_ts"] = pd.to_datetime(
    eligible["session_start_ts"],
    errors="coerce"
)

start_date = eligible["session_start_ts"].min()
end_date = eligible["session_start_ts"].max()

days_observed = (end_date - start_date).days

print("Start date:", start_date)
print("End date:", end_date)
print("Dataset spans (days):", days_observed)

Start date: 2025-10-21 11:46:23
End date: 2026-01-19 09:57:27
Dataset spans (days): 89


*Sensitivity Assumptions*

Because the experiment result was not statistically significant (p = 0.14), impact projections are modeled using scenario-based assumptions:

Worst case: only 50% of the observed lift materializes after rollout.

Base case: the full observed lift continues.

Best case: performance improves to 125% of the observed lift due to optimization and learning effects.

These scenarios provide a range of potential outcomes rather than relying on a single estimate.

In [17]:
# PART E — Impact Estimation (30-Day Projection)

# Assumptions
# 1. Dataset spans 89 days (observed from data)
# 2. Traffic volume remains stable
# 3. Observed conversion lift continues
# 4. Contribution margin assumed at 30%

days_observed = 89
margin_rate = 0.30

# Baseline Metrics
conv_A = ab.loc["A", "mean"]
conv_B = ab.loc["B", "mean"]

absolute_lift = conv_B - conv_A

total_eligible = eligible.shape[0]

# Daily and 30-day traffic
daily_eligible = total_eligible / days_observed
monthly_eligible = daily_eligible * 30

avg_order_value = orders["net_amount"].mean()

# Base Case (Observed Lift)
base_orders = monthly_eligible * absolute_lift
base_revenue = base_orders * avg_order_value
base_margin = base_revenue * margin_rate

# Sensitivity Analysis
# Worst Case: 50% of observed lift
worst_orders = monthly_eligible * (absolute_lift * 0.5)
worst_revenue = worst_orders * avg_order_value
worst_margin = worst_revenue * margin_rate

# Best Case: 125% of observed lift
best_orders = monthly_eligible * (absolute_lift * 1.25)
best_revenue = best_orders * avg_order_value
best_margin = best_revenue * margin_rate

# Output Results
print("--30-Day Impact Projection--")
print("Observed Days:", days_observed)
print("Estimated 30-Day Eligible Sessions:", round(monthly_eligible))

print("\n--Worst Case (50% of Observed Lift)--")
print("Incremental Orders:", round(worst_orders))
print("Incremental Revenue:", round(worst_revenue, 2))
print("Incremental Margin:", round(worst_margin, 2))

print("\n--Base Case (Observed Lift)--")
print("Incremental Orders:", round(base_orders))
print("Incremental Revenue:", round(base_revenue, 2))
print("Incremental Margin:", round(base_margin, 2))

print("\n--Best Case (125% of Observed Lift)--")
print("Incremental Orders:", round(best_orders))
print("Incremental Revenue:", round(best_revenue, 2))
print("Incremental Margin:", round(best_margin, 2))


--30-Day Impact Projection--
Observed Days: 89
Estimated 30-Day Eligible Sessions: 1002

--Worst Case (50% of Observed Lift)--
Incremental Orders: 8
Incremental Revenue: 63249.64
Incremental Margin: 18974.89

--Base Case (Observed Lift)--
Incremental Orders: 16
Incremental Revenue: 126499.27
Incremental Margin: 37949.78

--Best Case (125% of Observed Lift)--
Incremental Orders: 20
Incremental Revenue: 158124.09
Incremental Margin: 47437.23


### PART F — Export Clean Dataset for Tableau


In [18]:
# PART F — Export Clean Dataset for Tableau
dim_users = pd.read_csv("dim_users_enriched.csv")
export_df = eligible.merge(
    dim_users[["user_id", "segment"]],
    on="user_id",
    how="left"
)

export_columns = [
    "session_id",
    "session_start_ts",
    "variant",
    "device",
    "channel",
    "segment",
    "is_new_user",
    "has_productview",
    "has_addtocart",
    "has_begincheckout",
    "has_paymentattempt",
    "has_purchase",
    "net_amount"
]

export_df = export_df[export_columns]

export_df.to_csv("tableau_analysis_dataset.csv", index=False)

print("Export complete: tableau_analysis_dataset.csv")

Export complete: tableau_analysis_dataset.csv
